In [ ]:
!git clone https://github.com/para0107/Cluj-Road-Intelligence-System.git
%cd Cluj-Road-Intelligence-System
!pip install -r requirements.txt

In [ ]:
import shutil
import os

# Build the base folder structure
os.makedirs('data/datasets', exist_ok=True)

print("Copying RDD2022 (This will take a few minutes for 28k images)...")
# Using the exact path you found!
shutil.copytree('/kaggle/input/datasets/paraschiv/cluj-raw-datasets/rdd2022', 'data/datasets/rdd2022', dirs_exist_ok=True)

print("Copying Pothole-600...")
# Applying the same path structure to pothole600
shutil.copytree('/kaggle/input/datasets/paraschiv/cluj-raw-datasets/pothole600', 'data/datasets/pothole600', dirs_exist_ok=True)

print("Data successfully copied to writable space! Folder tree matches local setup.")

In [ ]:
# Move all contents from the nested rdd2022 folder up one level
!mv data/datasets/rdd2022/rdd2022/* data/datasets/rdd2022/

# Remove the empty nested folder to clean things up
!rm -rf data/datasets/rdd2022/rdd2022

print("RDD2022 folder flattened successfully!")

In [ ]:
!mv data/datasets/pothole600/pothole600/* data/datasets/pothole600/
!rm -rf data/datasets/pothole600/pothole600

In [ ]:
!python ml/detection/data_prep/prep_rdd2022.py --dataset_dir data/datasets/rdd2022
!python ml/detection/data_prep/prep_pothole600.py
!python ml/detection/data_prep/merge_datasets.py
!python ml/detection/data_prep/coco_to_yolo.py

In [ ]:
# Nuke the loggers so they don't crash the end of the epoch
!pip uninstall -y wandb ray

# Copy your Phase 1 weights so the script can find them
!cp /kaggle/input/datasets/paraschiv/cluj-road-weights/rtdetr-l.pt ./rtdetr-l.pt

# Start training, but force a safe stop at 11.5 hours!
!python ml/detection/train.py --freeze_epochs 0 --epochs 90 --device 0,1 --workers 4 --time 11.5

In [ ]:
import shutil
import os

print("Zipping the runs folder...")
shutil.make_archive('/kaggle/working/runs_backup', 'zip', 'runs')
print("Successfully zipped! Ready for download.")